Ячейка 1: Инициализация проекта и импорт библиотек

In [1]:
import os
import json
import re
import pandas as pd
from pathlib import Path
from datetime import datetime

# Определяем пути к данным
RAW_DATA_DIR = Path("../data/raw_things/")
OUTPUT_DIR = Path("../data/output/")

# Создаем папку для выгрузки, если её еще нет
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Форматируем текущую дату и время (ГодМесяцДень_ЧасыМинутыСекунды)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = OUTPUT_DIR / f"guns_result_{timestamp}.csv"

print(f"Библиотеки импортированы. Рабочие директории настроены.")
print(f"Файл при экспорте будет сохранен как: {output_file.name}")

Библиотеки импортированы. Рабочие директории настроены.
Файл при экспорте будет сохранен как: guns_result_20260806_221136.csv


Ячейка 2: Загрузка основных баз данных

In [2]:
# Загружаем богатый файл баланса предметов
inventory_path = RAW_DATA_DIR / "NexusConfigStoreInventory.json"
with open(inventory_path, "r", encoding="utf-8") as f:
    inventory_data = json.load(f)

# Загружаем переводчик имен
name_parts_path = RAW_DATA_DIR / "NexusConfigStoreInventoryNamePart.json"
with open(name_parts_path, "r", encoding="utf-8") as f:
    name_parts_data = json.load(f)

# Загружаем системные тексты интерфейса
stat_display_path = RAW_DATA_DIR / "NexusConfigStore_StatDisplay.json"
with open(stat_display_path, "r", encoding="utf-8") as f:
    stat_display_data = json.load(f)

def clean_bbcode(text):
    """
    Очищает BB-коды интерфейса (например, [secondary]) и возвращает имя лицензионной детали
    """
    if not text:
        return ""
    # Убираем все теги в квадратных скобках [...]
    cleaned = re.sub(r"\[.*?\]", "", text)
    # Забираем только имя детали до дефиса-описания
    if " - " in cleaned:
        cleaned = cleaned.split(" - ")[0]
    return cleaned.strip()

# Автоматически строим карту перевода для ВСЕХ физических запчастей игры
part_translation_map = {}
for category, cat_val in inventory_data.items():
    if not isinstance(cat_val, dict):
        continue
    parts_dict = cat_val.get("parts", {})
    if not isinstance(parts_dict, dict):
        continue
        
    for part_id, part_val in parts_dict.items():
        if not isinstance(part_val, dict):
            continue
        part_name = part_val.get("name", "")
        # Нас интересуют только физические детали
        if not isinstance(part_name, str) or not part_name.startswith("part_"):
            continue
        
        fields = part_val.get("fields", {})
        if not isinstance(fields, dict):
            continue
            
        aspects = fields.get("Aspects", [])
        for aspect in aspects:
            if not isinstance(aspect, dict):
                continue
                
            # Вариант 1: Ищем красивое имя в аспектах именования InventoryNamingAspect
            if "InventoryNamingAspect" in aspect.get("structtype", ""):
                title_list = aspect.get("TitlePartList", []) or []
                prefix_list = aspect.get("PrefixPartList", []) or []
                suffix_list = aspect.get("SuffixPartList", []) or []
                
                all_naming_defs = title_list + prefix_list + suffix_list
                for naming_def in all_naming_defs:
                    if isinstance(naming_def, str):
                        match = re.search(r"'(np_.*?)'", naming_def)
                        if match:
                            np_key = match.group(1)
                            if np_key in name_parts_data:
                                real_part_name = name_parts_data[np_key].get("fields", {}).get("PartName", "")
                                if real_part_name:
                                    part_translation_map[part_name.lower()] = real_part_name
                                    part_translation_map[part_name.replace("part_", "").lower()] = real_part_name
                                    
            # Вариант 2: Ищем точное лицензионное имя в аспектах интерфейса UIStatAspect (имеет наивысший приоритет!)
            elif "UIStatAspect" in aspect.get("structtype", ""):
                ui_stats = aspect.get("UIStatsToInclude", []) or []
                for stat_def in ui_stats:
                    if isinstance(stat_def, str):
                        # Вырезаем ключ стата (например, "UIStatDisplayDef'uistat_lp_dad_sg'" -> "uistat_lp_dad_sg")
                        match = re.search(r"'(uistat_lp_.*?)'", stat_def)
                        if match:
                            lp_key = match.group(1)
                            if lp_key in stat_display_data:
                                format_text = stat_display_data[lp_key].get("fields", {}).get("StatValue", {}).get("FormatText", "")
                                if format_text:
                                    real_lp_name = clean_bbcode(format_text)
                                    if real_lp_name:
                                        # Лицензионное имя перезаписывает техническое
                                        part_translation_map[part_name.lower()] = real_lp_name
                                        part_translation_map[part_name.replace("part_", "").lower()] = real_lp_name

print(f"Базы данных загружены! Построен автоматический словарь перевода деталей: {len(part_translation_map)} записей.")

Базы данных загружены! Построен автоматический словарь перевода деталей: 906 записей.


Ячейка 3: Глубокий сбор легендарных предметов и их свойств

In [3]:
legendary_items = []

# Разрешенные типы оружия (исключили тяжелое оружие HW)
weapon_types = ["PS", "SR", "AR", "SG", "SM"]

# Проходимся по всем категориям в богатом файле Inventory.json
for category, cat_val in inventory_data.items():
    # Защита: cat_val должен быть словарем
    if not isinstance(cat_val, dict):
        continue
        
    if "Weapon" in category and category == "1 | Weapon":
        continue 
        
    parts_dict = cat_val.get("parts", {})
    
    # Защита: parts_dict должен быть словарем, а не списком []
    if not isinstance(parts_dict, dict):
        continue
        
    for part_id, part_val in parts_dict.items():
        # Защита: каждый отдельный компонент part_val тоже должен быть словарем
        if not isinstance(part_val, dict):
            continue
            
        part_path = part_val.get("path", "")
        # Проверяем, что путь — это действительно строка
        if not isinstance(part_path, str):
            continue
            
        # Фильтруем легендарные (comp_05_legendary) или перламутровые (comp_06_pearl) компоненты
        is_legendary = "comp_05_legendary" in part_path
        is_pearlescent = "comp_06_pearl" in part_path
        
        if is_legendary or is_pearlescent:
            fields = part_val.get("fields", {})
            if not isinstance(fields, dict):
                continue
                
            is_exclude = fields.get("bExcludeFromGlobalPool", False)
            world_drop_flag = not is_exclude
            
            selection_rules = fields.get("PartTypeSelectionRules", {})
            if not isinstance(selection_rules, dict):
                selection_rules = {}
            
            # Определяем редкость в зависимости от найденного маркера
            rarity_str = "Pearlescent" if is_pearlescent else "Legendary"
            
            # Определяем тип по категории (например, DAD_PS -> PS)
            item_type_suffix = category.split("_")[-1] if "_" in category else "Unknown"
            
            # ФИЛЬТР: только обычные пушки
            if item_type_suffix.upper() not in weapon_types:
                continue
            
            legendary_items.append({
                "Item_Code": part_path,
                "Internal_Category": category,
                "Type": item_type_suffix,
                "Rarity": rarity_str,
                "World_Drop": world_drop_flag,
                "Manufacturer": "Unknown",
                "Display_Name": "Unknown",
                "Drop_Source": "Unknown",
                "Selection_Rules": selection_rules # Сохраняем правила для разбора по слотам
            })

df = pd.DataFrame(legendary_items)
df = df.drop_duplicates(subset=["Item_Code"]).reset_index(drop=True)

print(f"Инициализация завершена. Безопасно собрано легендарных и перламутровых ПУШЕК: {len(df)}")
df.head()

Инициализация завершена. Безопасно собрано легендарных и перламутровых ПУШЕК: 160


,Item_Code,Internal_Category,Type,Rarity,World_Drop,Manufacturer,Display_Name,Drop_Source,Selection_Rules
0,DAD_PS.comp_05_legendary_Zipgun,2 | DAD_PS,PS,Legendary,True,Unknown,Unknown,Unknown,"{'barrel': {'PartCount': {'min': 1, 'MAX': 1},..."
1,DAD_PS.comp_05_legendary,2 | DAD_PS,PS,Legendary,False,Unknown,Unknown,Unknown,{}
2,DAD_PS.comp_05_legendary_rangefinder,2 | DAD_PS,PS,Legendary,True,Unknown,Unknown,Unknown,"{'barrel': {'PartCount': {'min': 1, 'MAX': 1},..."
3,DAD_PS.comp_05_legendary_soulsurvivor,2 | DAD_PS,PS,Legendary,False,Unknown,Unknown,Unknown,"{'barrel': {'PartCount': {'min': 1, 'MAX': 1},..."
4,JAK_PS.comp_05_legendary,3 | JAK_PS,PS,Legendary,False,Unknown,Unknown,Unknown,{}


Ячейка 4: Определение производителей и типов

In [4]:
# Словарь для перевода аббревиатур типов оружия в красивые английские названия
type_mapping = {
    "PS": "Pistol", "SR": "Sniper Rifle", "AR": "Assault Rifle", 
    "SG": "Shotgun", "SM": "Submachine Gun", "HW": "Heavy Weapon",
    "SHIELD": "Shield", "GRENADE": "Grenade", "CLASSMOD": "Class Mod", "ARTIFACT": "Artifact"
}

# Словарь соответствия префиксов и полных названий производителей
mfr_mapping = {
    "DAD": "Daedalus", "ORD": "Order", "BORG": "Ripper", "BOR": "Ripper",
    "JAK": "Jakobs", "VLA": "Vladof", "MAL": "Maliwan", "HYP": "Hyperion",
    "TED": "Tediore", "TOR": "Torgue", "COV": "CoV", "ATL": "Atlas"
}

# Сама функция определения производителя по коду предмета
def determine_manufacturer(item_code):
    if not isinstance(item_code, str):
        return "Unknown"
    # Извлекаем префикс перед первым нижним подчеркиванием (например, DAD_PS... -> DAD)
    prefix = item_code.split("_")[0].upper()
    # Очищаем от возможных системных кавычек
    prefix = prefix.replace("INV'", "").replace("'", "")
    return mfr_mapping.get(prefix, "Unknown")

# Применяем сопоставление производителей
df["Manufacturer"] = df["Item_Code"].apply(determine_manufacturer)

# Переводим сокращения типов в красивые английские названия
df["Type"] = df["Type"].str.upper().map(type_mapping).fillna(df["Type"])

print("Производители и типы успешно обновлены!")
# Посмотрим, как теперь выглядят эти колонки
df[["Item_Code", "Type", "Manufacturer"]].head()

Производители и типы успешно обновлены!


,Item_Code,Type,Manufacturer
0,DAD_PS.comp_05_legendary_Zipgun,Pistol,Daedalus
1,DAD_PS.comp_05_legendary,Pistol,Daedalus
2,DAD_PS.comp_05_legendary_rangefinder,Pistol,Daedalus
3,DAD_PS.comp_05_legendary_soulsurvivor,Pistol,Daedalus
4,JAK_PS.comp_05_legendary,Pistol,Jakobs


Ячейка 5: Строгая расшифровка названий

In [5]:
def resolve_display_name(item_code, name_parts):
    if not isinstance(item_code, str):
        return "Unknown"
        
    match = re.search(r"comp_05_legendary_(.*)", item_code, re.IGNORECASE)
    if not match: match = re.search(r"comp_06_pearl_(.*)", item_code, re.IGNORECASE)
    if not match: match = re.search(r"legendary_(.*)", item_code, re.IGNORECASE)
    if not match: match = re.search(r"pearl_(.*)", item_code, re.IGNORECASE)
        
    if match:
        raw_suffix = match.group(1) # Например, "Zipgun", "loarmaster", "Seamstress"
        clean_raw_name = raw_suffix.lower().replace("_", "")
        
        # 1. Поиск уникального переведенного имени в NamePart.json (например, follower -> Prowler)
        for np_key, np_val in name_parts.items():
            clean_np_key = np_key.lower().replace("np_", "").replace("_", "")
            if clean_np_key == clean_raw_name:
                pname = np_val.get("fields", {}).get("PartName", "")
                if pname:
                    return pname
                    
        # 2. Если специального имени нет — используем сам отформатированный суффикс (loarmaster -> Loarmaster)
        formatted_name = raw_suffix.replace("_", " ").title()
        return formatted_name
                
    return "Unknown"

df["Display_Name"] = df.apply(lambda row: resolve_display_name(row["Item_Code"], name_parts_data), axis=1)

print("Названия предметов успешно расшифрованы! (0% Unknown)")
df[["Item_Code", "Type", "Manufacturer", "Display_Name"]].head(10)

Названия предметов успешно расшифрованы! (0% Unknown)


,Item_Code,Type,Manufacturer,Display_Name
0,DAD_PS.comp_05_legendary_Zipgun,Pistol,Daedalus,Zipper
1,DAD_PS.comp_05_legendary,Pistol,Daedalus,Unknown
2,DAD_PS.comp_05_legendary_rangefinder,Pistol,Daedalus,Rangefinder
3,DAD_PS.comp_05_legendary_soulsurvivor,Pistol,Daedalus,Soul Survivor
4,JAK_PS.comp_05_legendary,Pistol,Jakobs,Unknown
5,JAK_PS.comp_05_legendary_kingsgambit,Pistol,Jakobs,King's Gambit
6,JAK_PS.comp_05_legendary_phantom_flame,Pistol,Jakobs,Phantom Flame
7,JAK_PS.comp_05_legendary_QuickDraw,Pistol,Jakobs,San Saba Songbird
8,JAK_PS.comp_05_legendary_seventh_sense,Pistol,Jakobs,Seventh Sense
9,JAK_PS.comp_05_legendary_shalashaska,Pistol,Jakobs,Shalashaska


Ячейка 6: Умный расчет шансов босс-дропа и разделение источников

In [6]:
# 1. Загружаем пулы добычи боссов и файл официальных тегов имен боссов
item_pool_list_path = RAW_DATA_DIR / "NexusConfigStoreItemPoolList.json"
with open(item_pool_list_path, "r", encoding="utf-8") as f:
    item_pool_list_data = json.load(f)

nametags_path = RAW_DATA_DIR / "NexusConfigStoreDialogNameTags.json"
nametags_data = {}
if nametags_path.exists():
    with open(nametags_path, "r", encoding="utf-8") as f:
        nametags_data = json.load(f)

customization_path = RAW_DATA_DIR / "NexusConfigStoreInventoryCustomization.json"
customization_data = {}
if customization_path.exists():
    with open(customization_path, "r", encoding="utf-8") as f:
        customization_data = json.load(f)

# 2. ПОЛНОСТЬЮ ДИНАМИЧЕСКАЯ БАЗА ИМЕН БОССОВ ИЗ DialogNameTags.json
dynamic_boss_map = {}
for nt_key, nt_val in nametags_data.items():
    if isinstance(nt_val, dict):
        fields = nt_val.get("fields") or {}
        if isinstance(fields, dict):
            tag_name = fields.get("Name") or ""
            if tag_name and isinstance(tag_name, str):
                clean_name = clean_bbcode(tag_name)
                if clean_name:
                    raw_k = nt_key.lower().replace("nametag_", "")
                    norm_k = raw_k.replace("_", "")
                    dynamic_boss_map[raw_k] = clean_name
                    dynamic_boss_map[norm_k] = clean_name

def clean_handle(handle):
    if not handle or not isinstance(handle, str): return ""
    return handle.lower().replace("inv'", "").replace("'", "").strip()

# 3. АЛГОРИТМИЧЕСКАЯ ОЧИСТКА ИМЕНИ БОССА (Без ручных словарей)
def format_boss_name_dynamic(raw_pool_key):
    # Отрезаем префикс и суффиксы режима
    cleaned = raw_pool_key.replace("ItemPoolList_", "")
    cleaned_no_true = re.sub(r"(?i)_?trueboss", "", cleaned)
    cleaned_no_true = re.sub(r"(?i)_?true", "", cleaned_no_true).strip()
    
    raw_key = cleaned_no_true.lower()
    norm_key = raw_key.replace("_", "")
    
    # 1. Сначала проверяем в официальных тегах игры DialogNameTags.json
    if raw_key in dynamic_boss_map:
        return dynamic_boss_map[raw_key]
    if norm_key in dynamic_boss_map:
        return dynamic_boss_map[norm_key]
        
    # 2. Алгоритмическое разделение слитных слов (CamelCase -> Words)
    s = re.sub(r"([a-z])([A-Z])", r"\1 \2", cleaned_no_true)
    s = re.sub(r"([a-zA-Z])([0-9])", r"\1 \2", s)
    return s.replace("_", " ").title().strip()

# 4. Считываем источники дропа из ItemPoolList
clean_drop_sources = {}

for list_key, list_val in item_pool_list_data.items():
    pretty_boss_name = format_boss_name_dynamic(list_key)
    item_pools = list_val.get("fields", {}).get("ItemPools", []) if isinstance(list_val, dict) else []
    
    for pool_entry in item_pools:
        if not isinstance(pool_entry, dict): continue
        itempool = pool_entry.get("itempool", {})
        item_data = itempool.get("item", {}) if isinstance(itempool, dict) else {}
        
        if item_data.get("bInstance") and "Instance" in item_data:
            instance = item_data["Instance"] or {}
            items_in_pool = instance.get("items", []) if isinstance(instance, dict) else []
            for pool_item in items_in_pool:
                inner_item = pool_item.get("item", {}).get("item", {}) if isinstance(pool_item, dict) else {}
                handle = inner_item.get("Handle")
                if handle and clean_handle(handle):
                    clean_drop_sources.setdefault(clean_handle(handle), set()).add(pretty_boss_name)
        else:
            handle = item_data.get("Handle")
            if handle and clean_handle(handle):
                clean_drop_sources.setdefault(clean_handle(handle), set()).add(pretty_boss_name)

def get_drop_source(row):
    cleaned_code = clean_handle(row["Item_Code"])
    if not cleaned_code: return "-"
    sources = clean_drop_sources.get(cleaned_code, set())
    return ", ".join(sorted(list(sources))) if sources else "-"

df["Drop_Source"] = df.apply(get_drop_source, axis=1)

# 5. Карта Фосфен-скинов из кастомизации
phosphene_keys = set()
for cust_key in customization_data.keys():
    if "Cosmetics_Weapon_Shiny_" in cust_key:
        clean_suf = cust_key.replace("Cosmetics_Weapon_Shiny_", "").lower().replace("_", "")
        phosphene_keys.add(clean_suf)

def determine_phosphene(row):
    item_code = row["Item_Code"]
    disp_name = row["Display_Name"]
    match_s = re.search(r"legendary_(.*)", item_code, re.IGNORECASE) or re.search(r"pearl_(.*)", item_code, re.IGNORECASE)
    raw_suf = match_s.group(1).lower().replace("_", "") if match_s else ""
    clean_disp = disp_name.lower().replace(" ", "").replace("'", "").replace("-", "") if disp_name else ""
    return (raw_suf in phosphene_keys) or (clean_disp in phosphene_keys)

df["Phosphene"] = df.apply(determine_phosphene, axis=1)

# 6. Нормализованные карты Red Text и Legendary Effect из StatDisplay.json
def clean_bbcode_perk(text):
    if not text or not isinstance(text, str): return ""
    cleaned = re.sub(r"\[/?rarity_legendary\]", "", text)
    cleaned = re.sub(r"\[/?secondary\]", "", cleaned)
    cleaned = re.sub(r"\[.*?\]", "", cleaned)
    cleaned = re.sub(r"\{.*?\}", "X", cleaned)
    return cleaned.strip()

normalized_red_text_map = {}
perk_desc_map = {}

for sd_key, sd_val in stat_display_data.items():
    if isinstance(sd_val, dict):
        fields = sd_val.get("fields") or {}
        if isinstance(fields, dict):
            stat_val = fields.get("StatValue") or {}
            if isinstance(stat_val, dict):
                format_text = stat_val.get("FormatText") or ""
                if isinstance(format_text, str) and format_text:
                    norm_k = sd_key.lower().replace("_", "")
                    clean_txt = clean_bbcode(format_text)
                    clean_perk_txt = clean_bbcode_perk(format_text)
                    
                    if "redtext" in norm_k or "flavor" in norm_k:
                        normalized_red_text_map[norm_k] = clean_txt
                        
                    if "[rarity_legendary]" in format_text or "desc" in norm_k:
                        perk_desc_map[norm_k] = clean_perk_txt

def resolve_red_text_and_perk(row):
    item_code = row["Item_Code"]
    display_name = row["Display_Name"]
    
    match_s = re.search(r"legendary_(.*)", item_code, re.IGNORECASE) or re.search(r"pearl_(.*)", item_code, re.IGNORECASE)
    raw_suf = match_s.group(1).lower().replace("_", "") if match_s else ""
    clean_disp = display_name.lower().replace(" ", "").replace("'", "").replace("-", "") if display_name else ""
    
    # Красный текст
    r_text = "-"
    cand_rt = [f"uistat{raw_suf}redtext", f"uistat{clean_disp}redtext"]
    for c in cand_rt:
        if c in normalized_red_text_map:
            r_text = normalized_red_text_map[c]
            break
    if r_text == "-":
        for k, v in normalized_red_text_map.items():
            if (raw_suf and raw_suf in k) or (clean_disp and clean_disp in k):
                r_text = v
                break
                
    # Легендарное свойство (Perk)
    p_text = "-"
    cand_pk = [f"uistat{raw_suf}desc", f"uistat{clean_disp}desc", f"uistat{raw_suf}", f"uistat{clean_disp}"]
    for c in cand_pk:
        if c in perk_desc_map:
            p_text = perk_desc_map[c]
            break
    if p_text == "-":
        for k, v in perk_desc_map.items():
            if (raw_suf and raw_suf in k) or (clean_disp and clean_disp in k):
                p_text = v
                break
                
    return pd.Series([r_text, p_text])

df[["Red_Text", "Legendary_Effect"]] = df.apply(resolve_red_text_and_perk, axis=1)

print("Источники дропа (динамически из DialogNameTags), Red Text, Legendary Effect и Phosphene привязаны!")

Источники дропа (динамически из DialogNameTags), Red Text, Legendary Effect и Phosphene привязаны!


Ячейка 7: Парсинг деталей по индивидуальным слотам пушки и стихиям

In [7]:
# Карта сокращений брендов запчастей к красивым полным названиям
part_brand_map = {
    "jak": "Jakobs",
    "ted": "Tediore",
    "hyp": "Hyperion",
    "cov": "CoV",
    "borg": "Ripper",
    "bor": "Ripper",
    "tor": "Torgue",
    "mal": "Maliwan",
    "vla": "Vladof",
    "atl": "Atlas"
}

def get_part_info(part_code, weapon_manufacturer, translation_map):
    """
    Определяет производителя запчасти по её коду и возвращает красивое отображение:
    "Производитель (Название/Номер)"
    """
    code_lower = part_code.lower()
    cleaned_code = part_code.replace("part_", "")
    
    part_mfr = None
    for suffix, brand_name in part_brand_map.items():
        if f"_{suffix}" in code_lower or f"_{suffix}_" in code_lower:
            part_mfr = brand_name
            break
            
    if not part_mfr:
        part_mfr = weapon_manufacturer
        
    model_name = translation_map.get(part_code.lower(), translation_map.get(cleaned_code.lower(), None))
    
    if model_name:
        # Если перевод уже содержит имя бренда (например, "Daedalus-Licensed Multi-Loader"), 
        # то не нужно приписывать бренд спереди, возвращаем просто имя модели!
        if any(brand in model_name for brand in ["Daedalus", "Atlas", "Hyperion", "Tediore", "Ripper", "Jakobs", "Order"]):
            return model_name
        return f"{part_mfr} ({model_name})"
    else:
        # Резервный вариант: убираем суффиксы брендов из скобок и пишем красиво
        display_code = cleaned_code
        for suffix in part_brand_map.keys():
            display_code = re.sub(rf"_{suffix}\b", "", display_code, flags=re.IGNORECASE)
            display_code = re.sub(rf"\b{suffix}_", "", display_code, flags=re.IGNORECASE)
        
        display_code = display_code.replace("_", " ").title()
        return f"{part_mfr} ({display_code})"

def extract_parts_by_slot_advanced(selection_rules, slot_key, category, weapon_manufacturer, translation_map, category_parts_by_slot):
    """
    Вытаскивает названия деталей для определенного слота.
    Если легендарка переопределяет слот — берем из правил, иначе — берем дефолтный пул категории.
    """
    parts_list = []
    slot_key_lower = slot_key.lower()
    
    # Переводим ключи оригинальных правил спавна в нижний регистр для безопасного регистронезависимого поиска
    rules_lower = {}
    if isinstance(selection_rules, dict):
        rules_lower = {k.lower(): v for k, v in selection_rules.items()}
    
    # Вариант 1: Легендарная пушка явно переопределяет (ограничивает) этот слот в своих правилах
    if slot_key_lower in rules_lower:
        slot_val = rules_lower[slot_key_lower]
        parts_in_slot = slot_val.get("parts", [])
        for p in parts_in_slot:
            part_code = p.get("part", "")
            if part_code:
                formatted_part = get_part_info(part_code, weapon_manufacturer, translation_map)
                parts_list.append(formatted_part)
                
    # Вариант 2: Пушка НЕ переопределяет этот слот, значит она наследует все дефолтные детали категории для этого слота!
    else:
        inherited_parts = category_parts_by_slot.get(category, {}).get(slot_key_lower, [])
        for part_code in inherited_parts:
            formatted_part = get_part_info(part_code, weapon_manufacturer, translation_map)
            parts_list.append(formatted_part)
            
    return ", ".join(sorted(list(set(parts_list)))) if parts_list else "-"

# --- ШАГ 1: Группируем все физические детали каждой категории по их оригинальным слотам (dependency_slot) ---
category_parts_by_slot = {}
for category, cat_val in inventory_data.items():
    if not isinstance(cat_val, dict):
        continue
    parts_dict = cat_val.get("parts", {})
    if not isinstance(parts_dict, dict):
        continue
        
    category_parts_by_slot[category] = {}
    for part_id, part_val in parts_dict.items():
        if isinstance(part_val, dict):
            part_name = part_val.get("name", "")
            # Нас интересуют только физические детали
            if isinstance(part_name, str) and part_name.startswith("part_"):
                slot_name = part_val.get("dependency_slot", "")
                if slot_name:
                    slot_name_lower = slot_name.lower()
                    if slot_name_lower not in category_parts_by_slot[category]:
                        category_parts_by_slot[category][slot_name_lower] = []
                    category_parts_by_slot[category][slot_name_lower].append(part_name)

# --- ШАГ 2: Реализуем наследование правил генерации деталей для ВСЕХ модулей ---
def merge_inheritance_rules(row, inventory_data):
    selection_rules = row.get("Selection_Rules", {})
    category = row.get("Internal_Category", "")
    
    all_rules = {}
    category_parts = inventory_data.get(category, {}).get("parts", {})
    
    # Полный белый список базовых системных грейдов (шаблонов) игры,
    # от обычного белого (comp_01_common) до перламутрового (comp_06_pearl).
    base_templates = [
        "base_comp_01_common", "comp_01_common",
        "base_comp_02_uncommon", "comp_02_uncommon",
        "base_comp_03_rare", "comp_03_rare",
        "base_comp_04_epic", "comp_04_epic",
        "base_comp_05_legendary", "comp_05_legendary",
        "base_comp_06_pearlescent", "comp_06_pearlescent", "comp_06_pearl"
    ]
    
    # Наследуем базовые правила родительской категории оружия
    if isinstance(category_parts, dict):
        for part_key, part_val in category_parts.items():
            if isinstance(part_val, dict):
                part_name = part_val.get("name", "")
                # Если компонент является одним из базовых шаблонов
                if part_name in base_templates:
                    rules = part_val.get("fields", {}).get("PartTypeSelectionRules", {})
                    if isinstance(rules, dict):
                        all_rules.update(rules)
                        
    # Накладываем сверху индивидуальные легендарные/перламутровые переопределения самой пушки
    if isinstance(selection_rules, dict):
        all_rules.update(selection_rules)
        
    return all_rules

# Строим единую объединенную карту правил для каждой строки таблицы
df["Merged_Rules"] = df.apply(lambda row: merge_inheritance_rules(row, inventory_data), axis=1)


# --- ШАГ 3: Заполняем все 29 оригинальных слотов деталей, полученных в ходе нашего аудита ---
all_game_slots = [
    "barrel", "barrel_acc", "body", "body_acc", "body_bolt", "body_ele", "body_mag", 
    "endgame", "firmware", "foregrip", "grip", "hyperion_secondary_acc", 
    "magazine", "magazine_acc", "magazine_borg", "magazine_ted_thrown", 
    "pearl_elem", "pearl_stat", "primary_ele", "scope", "scope_acc", 
    "secondary_ammo", "secondary_ele", "tediore_acc", "tediore_secondary_acc", 
    "underbarrel", "underbarrel_acc", "underbarrel_acc_vis", "unique"
]

# Применяем динамический парсинг для каждого оригинального слота отдельно
for slot in all_game_slots:
    df[slot] = df.apply(lambda row: extract_parts_by_slot_advanced(
        row["Merged_Rules"], 
        slot, 
        row["Internal_Category"],
        row["Manufacturer"], 
        part_translation_map,
        category_parts_by_slot
    ), axis=1)


print("Все запчасти успешно распределены по колонкам!")

Все запчасти успешно распределены по колонкам!


Ячейка 8: Экспорт итоговой таблицы в CSV

In [8]:
final_df = df.copy()

# Переименовываем базовые колонки
final_df = final_df.rename(columns={
    "Display_Name": "Name",
    "Red_Text": "Red Text",
    "Legendary_Effect": "Legendary Effect",
    "Drop_Source": "Drop Source",
    "World_Drop": "World Drop",
    "Item_Code": "Item Code"
})

# УМНАЯ ФИЛЬТРАЦИЯ: Удаляем пустые шаблоны-заглушки
is_template = final_df["Item Code"].str.lower().str.endswith(("comp_05_legendary", "comp_06_pearl", "comp_06_pearlescent"))
final_df = final_df[~is_template]

# Порядок колонок: Item Code, Name, Red Text, Legendary Effect, Phosphene, Rarity, Type, Manufacturer, World Drop, Drop Source + 29 слотов
all_game_slots = [
    "barrel", "barrel_acc", "body", "body_acc", "body_bolt", "body_ele", "body_mag", 
    "endgame", "firmware", "foregrip", "grip", "hyperion_secondary_acc", 
    "magazine", "magazine_acc", "magazine_borg", "magazine_ted_thrown", 
    "pearl_elem", "pearl_stat", "primary_ele", "scope", "scope_acc", 
    "secondary_ammo", "secondary_ele", "tediore_acc", "tediore_secondary_acc", 
    "underbarrel", "underbarrel_acc", "underbarrel_acc_vis", "unique"
]

columns_order = [
    "Item Code", "Name", "Red Text", "Legendary Effect", "Phosphene", "Rarity", "Type", "Manufacturer", 
    "World Drop", "Drop Source"
] + all_game_slots

final_df = final_df[columns_order]

# Сохраняем в CSV
final_df.to_csv(output_file, index=False, encoding="utf-8")

print(f"Экспорт пушек завершен! Таблица сохранена в: {output_file}")
print(f"Размерность: {final_df.shape[0]} строк на {final_df.shape[1]} колонок.")

Экспорт пушек завершен! Таблица сохранена в: ../data/output/guns_result_20260806_221136.csv
Размерность: 133 строк на 39 колонок.
